# 🤖 Task 3 – Resume / Candidate Screening System
**Future Interns | Machine Learning Track**

---
### 🎯 Objective
Build an ML system to automatically screen and rank resumes based on a given job role using NLP techniques.

### 🔧 Key Features
- Resume text cleaning & parsing
- Skill extraction & matching with job descriptions
- Candidate ranking based on role fit (TF-IDF + Cosine Similarity)
- Skill gap identification

### 📦 Tools Used
`Python` | `NLTK` | `Scikit-learn` | `Pandas` | `Matplotlib` | `Seaborn`

## 📥 Step 1 – Install & Import Libraries

In [ ]:
# Install required libraries (run once)
# !pip install nltk scikit-learn pandas matplotlib seaborn python-docx PyMuPDF

import re
import string
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ All libraries imported successfully!")

## 📄 Step 2 – Define Resume Dataset
> **Note:** In a real system, resumes would be parsed from `.pdf` or `.docx` files. Here we use structured text. The first resume is based on the actual uploaded resume of the candidate.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Resume Dataset  (Candidate 1 = Wasim's actual uploaded resume)
# ─────────────────────────────────────────────────────────────────
resumes = [
    {
        "name": "Mohd Wasim",
        "text": """
        Beginner in Machine Learning, currently learning and building projects to gain practical experience.
        Skills: Python, Basic Machine Learning, Pandas, NumPy, Data Cleaning.
        Projects: Sales and Demand Forecasting - Built ML model using Linear Regression and Random Forest,
        performed data preprocessing and prediction.
        Education: B.Tech Mechanical Engineering, Zakir Husain College of Engineering and Technology.
        Strengths: Quick learner, Consistent, Interested in AI ML.
        GitHub: https://github.com/mohdwasim08
        """
    },
    {
        "name": "Priya Sharma",
        "text": """
        Data Scientist with 2 years of experience in machine learning, deep learning, and NLP.
        Skills: Python, TensorFlow, PyTorch, Scikit-learn, Pandas, NumPy, Matplotlib, SQL, Docker.
        Projects: Sentiment Analysis using BERT, Image Classification using CNN, Customer Churn Prediction.
        Education: M.Tech Computer Science, IIT Delhi.
        Certifications: Google ML Crash Course, Coursera Deep Learning Specialization.
        """
    },
    {
        "name": "Arjun Mehta",
        "text": """
        Full Stack Web Developer with 3 years of experience in React, Node.js, and databases.
        Skills: JavaScript, React, Node.js, Express, MongoDB, MySQL, HTML, CSS, REST APIs, Git.
        Projects: E-commerce website, Blog platform, Real-time chat application using Socket.io.
        Education: B.Tech Information Technology, VIT University.
        """
    },
    {
        "name": "Sneha Patel",
        "text": """
        ML Engineer with experience in deploying scalable machine learning models.
        Skills: Python, Scikit-learn, XGBoost, LightGBM, Pandas, NumPy, Flask, FastAPI, Docker, Kubernetes, AWS.
        Projects: Fraud Detection system using ensemble models, NLP pipeline for text classification,
        A/B testing framework for model evaluation.
        Education: B.Tech Computer Science, BITS Pilani.
        Experience: 1.5 years at a FinTech startup.
        """
    },
    {
        "name": "Rahul Gupta",
        "text": """
        Data Analyst with expertise in visualization and statistical analysis.
        Skills: Python, R, SQL, Tableau, Power BI, Excel, Pandas, Matplotlib, Seaborn, Statistics.
        Projects: Sales Dashboard using Tableau, Customer Segmentation using K-Means clustering,
        Exploratory Data Analysis on COVID-19 dataset.
        Education: B.Sc Statistics, Delhi University.
        """
    },
]

df = pd.DataFrame(resumes)
print(f"✅ Loaded {len(df)} resumes")
df[['name']]

## 🧹 Step 3 – Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Clean and normalize resume text."""
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove punctuation & digits
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['cleaned_text'] = df['text'].apply(preprocess_text)

print("✅ Text preprocessing complete!")
print("\n📋 Sample cleaned resume (Mohd Wasim):")
print(df['cleaned_text'][0][:300], '...')

## 💼 Step 4 – Define Job Description

In [ ]:
# ── Change this to any job role you want to screen for ──
job_description = """
We are looking for a Machine Learning Engineer / Data Scientist.
Required Skills: Python, Machine Learning, Scikit-learn, Pandas, NumPy, Data Cleaning,
Data Preprocessing, Model Evaluation, Regression, Classification, NLP.
Nice to have: TensorFlow, PyTorch, SQL, Docker, Flask, API development.
Experience with real-world ML projects and GitHub portfolio preferred.
Education: B.Tech or M.Tech in CS, IT, or related field.
"""

cleaned_jd = preprocess_text(job_description)

print("✅ Job Description loaded!")
print("\n📋 Cleaned JD:")
print(cleaned_jd)

## 🔍 Step 5 – TF-IDF Vectorization & Cosine Similarity Scoring

In [ ]:
# Combine JD + all resumes for TF-IDF fitting
all_texts = [cleaned_jd] + df['cleaned_text'].tolist()

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(all_texts)

# JD is index 0; resumes are index 1 onwards
jd_vector    = tfidf_matrix[0]
resume_vecs  = tfidf_matrix[1:]

# Cosine similarity between JD and each resume
scores = cosine_similarity(jd_vector, resume_vecs).flatten()

df['similarity_score'] = scores
df['match_percent']    = (scores * 100).round(2)

# Rank candidates
df_ranked = df[['name', 'similarity_score', 'match_percent']].sort_values(
    'similarity_score', ascending=False).reset_index(drop=True)
df_ranked.index += 1  # Rank starts at 1
df_ranked.index.name = 'Rank'

print("✅ Ranking complete!\n")
print(df_ranked.to_string())

## 🏷️ Step 6 – Priority Tagging

In [ ]:
def assign_priority(score):
    if score >= 0.45:
        return '🟢 High'
    elif score >= 0.25:
        return '🟡 Medium'
    else:
        return '🔴 Low'

df['priority'] = df['similarity_score'].apply(assign_priority)

result = df[['name', 'match_percent', 'priority']].sort_values(
    'match_percent', ascending=False).reset_index(drop=True)
result.index += 1
result.index.name = 'Rank'

print("📊 Final Candidate Screening Report")
print("=" * 50)
print(result.to_string())

## 🧩 Step 7 – Skill Extraction & Gap Analysis

In [ ]:
# Define required and nice-to-have skills from the JD
required_skills = [
    'python', 'machine learning', 'scikit-learn', 'pandas', 'numpy',
    'data cleaning', 'preprocessing', 'model evaluation', 'regression', 'classification'
]
bonus_skills = ['tensorflow', 'pytorch', 'sql', 'docker', 'flask', 'nlp', 'api']

def extract_skills(resume_text, skill_list):
    text_lower = resume_text.lower()
    found = [s for s in skill_list if s in text_lower]
    missing = [s for s in skill_list if s not in text_lower]
    return found, missing

print("🔍 Skill Gap Analysis per Candidate")
print("=" * 60)

gap_records = []
for _, row in df.iterrows():
    found_req, missing_req = extract_skills(row['text'], required_skills)
    found_bon, _           = extract_skills(row['text'], bonus_skills)
    coverage = round(len(found_req) / len(required_skills) * 100, 1)

    gap_records.append({
        'Candidate': row['name'],
        'Required Skills Found': len(found_req),
        'Required Skills Missing': len(missing_req),
        'Bonus Skills': len(found_bon),
        'Coverage %': coverage,
        'Missing Skills': ', '.join(missing_req) if missing_req else 'None'
    })

    print(f"\n👤 {row['name']}")
    print(f"   ✅ Found     : {', '.join(found_req) or 'None'}")
    print(f"   ❌ Missing   : {', '.join(missing_req) or 'None'}")
    print(f"   ⭐ Bonus     : {', '.join(found_bon) or 'None'}")
    print(f"   📊 Coverage  : {coverage}%")

gap_df = pd.DataFrame(gap_records)

## 📊 Step 8 – Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Resume Candidate Screening Dashboard', fontsize=18, fontweight='bold', y=1.01)

# ── 1. Similarity Score Bar Chart ──
ax1 = axes[0, 0]
sorted_df = df.sort_values('match_percent', ascending=True)
colors = ['#2ecc71' if p >= 45 else '#f39c12' if p >= 25 else '#e74c3c'
          for p in sorted_df['match_percent']]
bars = ax1.barh(sorted_df['name'], sorted_df['match_percent'], color=colors, edgecolor='white')
ax1.set_xlabel('Match Score (%)', fontsize=11)
ax1.set_title('Candidate Match Score vs Job Description', fontsize=13, fontweight='bold')
ax1.set_xlim(0, 100)
for bar, val in zip(bars, sorted_df['match_percent']):
    ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')
patches = [mpatches.Patch(color='#2ecc71', label='High (≥45%)'),
           mpatches.Patch(color='#f39c12', label='Medium (25-44%)'),
           mpatches.Patch(color='#e74c3c', label='Low (<25%)')]
ax1.legend(handles=patches, loc='lower right', fontsize=9)

# ── 2. Skill Coverage Heatmap ──
ax2 = axes[0, 1]
skill_matrix = []
for _, row in df.iterrows():
    row_skills = [1 if s in row['text'].lower() else 0 for s in required_skills]
    skill_matrix.append(row_skills)

skill_df = pd.DataFrame(skill_matrix, index=df['name'],
                         columns=[s.replace(' ', '\n') for s in required_skills])
sns.heatmap(skill_df, ax=ax2, cmap='RdYlGn', cbar=False,
            linewidths=0.5, linecolor='gray',
            annot=True, fmt='d', annot_kws={'size': 9})
ax2.set_title('Skill Coverage Heatmap (1=Present, 0=Missing)', fontsize=12, fontweight='bold')
ax2.set_xticklabels(ax2.get_xticklabels(), fontsize=8)
ax2.set_yticklabels(ax2.get_yticklabels(), fontsize=9, rotation=0)

# ── 3. Priority Distribution Pie Chart ──
ax3 = axes[1, 0]
priority_counts = df['priority'].value_counts()
clean_labels = [p.split(' ', 1)[1] for p in priority_counts.index]
pie_colors = []
for p in priority_counts.index:
    if 'High' in p:   pie_colors.append('#2ecc71')
    elif 'Medium' in p: pie_colors.append('#f39c12')
    else:              pie_colors.append('#e74c3c')
ax3.pie(priority_counts, labels=clean_labels, autopct='%1.0f%%',
        colors=pie_colors, startangle=90,
        textprops={'fontsize': 11}, pctdistance=0.75)
ax3.set_title('Candidate Priority Distribution', fontsize=13, fontweight='bold')

# ── 4. Required vs Bonus Skills Grouped Bar ──
ax4 = axes[1, 1]
x = np.arange(len(gap_df))
width = 0.35
b1 = ax4.bar(x - width/2, gap_df['Required Skills Found'], width,
              label='Required Skills Found', color='#3498db', edgecolor='white')
b2 = ax4.bar(x + width/2, gap_df['Bonus Skills'], width,
              label='Bonus Skills Found', color='#9b59b6', edgecolor='white')
ax4.set_xticks(x)
ax4.set_xticklabels(gap_df['Candidate'], fontsize=9)
ax4.set_ylabel('Skill Count')
ax4.set_title('Required vs Bonus Skills per Candidate', fontsize=13, fontweight='bold')
ax4.legend(fontsize=9)
ax4.set_ylim(0, max(gap_df['Required Skills Found'].max(), gap_df['Bonus Skills'].max()) + 2)
for bar in list(b1) + list(b2):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(int(bar.get_height())), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('screening_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard saved as screening_dashboard.png")

## 📋 Step 9 – Final Screening Report

In [ ]:
print("=" * 65)
print("      📄 FINAL CANDIDATE SCREENING REPORT")
print("      Role: Machine Learning Engineer / Data Scientist")
print("=" * 65)

final = df[['name', 'match_percent', 'priority']].sort_values(
    'match_percent', ascending=False).reset_index(drop=True)
final.index += 1
final.index.name = 'Rank'
final.columns = ['Candidate', 'Match %', 'Priority']

print(final.to_string())
print()

print("📌 Skill Gap Summary:")
print("-" * 65)
gap_summary = gap_df[['Candidate', 'Coverage %', 'Missing Skills']].set_index('Candidate')
print(gap_summary.to_string())
print()

# Top pick
top = df.loc[df['match_percent'].idxmax()]
print(f"\n🏆 Top Recommended Candidate: {top['name']} "
      f"({top['match_percent']:.1f}% match | {top['priority']})")

# Mohd Wasim's summary
wasim = df[df['name'] == 'Mohd Wasim'].iloc[0]
print(f"\n👤 Your Profile (Mohd Wasim):")
print(f"   Match Score : {wasim['match_percent']:.1f}%")
print(f"   Priority    : {wasim['priority']}")

wasim_found, wasim_missing = extract_skills(wasim['text'], required_skills)
print(f"   Found Skills: {', '.join(wasim_found) or 'None'}")
print(f"   Skill Gaps  : {', '.join(wasim_missing) or 'None'}")
print(f"\n💡 Tip: Adding skills like '{', '.join(wasim_missing[:3])}' "
      f"would significantly boost your match score!")

## ✅ Summary

| Step | Action |
|------|--------|
| 1 | Loaded and structured 5 candidate resumes |
| 2 | Cleaned text using NLTK (stopword removal, lemmatization) |
| 3 | Vectorized resumes and JD using TF-IDF |
| 4 | Computed Cosine Similarity scores for ranking |
| 5 | Assigned priority labels (High / Medium / Low) |
| 6 | Performed skill gap analysis per candidate |
| 7 | Visualized results with a 4-panel dashboard |

---
### 🔗 GitHub Repository Format
This notebook should be pushed to: **`FUTURE_ML_03`**

**Author:** Mohd Wasim | **Internship:** Future Interns – Machine Learning Track